#### Semantic Caching

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
cached_queries = []
cached_answers = []

vector_store = None

In [ ]:
class State(TypedDict):

    query: str
    answer: str

In [ ]:
# fake llm node
def call_llm(query: str) -> str:

    print("CALLING LLM...")

    return f"LLM answer for: {query}"

In [ ]:

def semantic_cache_node(state: State):

    global vector_store

    query = state["query"]

    # No cache yet
    if vector_store is None:

        answer = call_llm(query)

        cached_queries.append(query)
        cached_answers.append(answer)

        vector_store = FAISS.from_texts(
            cached_queries,
            embeddings
        )

        return {
            "answer": answer
        }

    # Search similar query
    results = vector_store.similarity_search_with_score(
        query,
        k=1
    )

    document, distance = results[0]

    print("Distance:", distance)

    # FAISS distance: smaller = more similar
    threshold = 0.5  ## can be stricter 

    if distance < threshold:

        index = cached_queries.index(
            document.page_content
        )

        print("SEMANTIC CACHE HIT")

        return {
            "answer": cached_answers[index]
        }

    # Cache miss
    print("SEMANTIC CACHE MISS")

    answer = call_llm(query)

    # Store new query
    cached_queries.append(query)
    cached_answers.append(answer)

    vector_store.add_texts(
        [query]
    )

    return {
        "answer": answer
    }

- Graph

In [ ]:
graph = StateGraph(State)

graph.add_node(
    "semantic_cache",
    semantic_cache_node
)

graph.add_edge(
    START,
    "semantic_cache"
)

graph.add_edge(
    "semantic_cache",
    END
)

app = graph.compile()

- output

In [ ]:
result1 = app.invoke({
    "query": "What is Parkinson's disease?"
})

print("\nAnswer 1:")
print(result1["answer"])


result2 = app.invoke({
    "query": "Can you explain Parkinson's disease?"
})

print("\nAnswer 2:")
print(result2["answer"])


result3 = app.invoke({
    "query": "What is the capital of France?"
})

print("\nAnswer 3:")
print(result3["answer"])